# Taxonomic Relevance Evaluation

This notebook evaluates a problem in the current species scoring setup: the manual annotations often encode dataset relevance through coarse taxonomic group and richness information, while the extraction output often enumerates detailed taxa. That creates large numbers of false positives and false negatives even when the extraction is arguably useful for screening dataset relevance.

A few concrete patterns illustrate the mismatch:

- **Count signal in manual annotation, enumerated taxa in prediction:** ground truth may say `73 weevil species`, while the model returns a long list of 73 named weevils. Raw `species` comparison treats that as a bad miss even though the prediction captures most of the screening signal.
- **Coarse group in manual annotation, scientific names in prediction:** ground truth may say `Caribou`, while the model returns `Rangifer tarandus` plus a subspecies or population label. This is partly a naming mismatch, not necessarily a relevance failure.
- **Broad taxonomic basket in manual annotation, mixed detailed taxa in prediction:** ground truth may record `12 mammal, 199 ground-dwelling beetles, 240 flying beetles`, while the model predicts many specific mammals and beetles. The prediction may be directionally useful, but not in the same representation as the annotation.
- **Community label in manual annotation, organism list in prediction:** ground truth may say `benthic intertidal community`, while the model outputs annelids, molluscs, and algae. That captures biological content, but only some derived views can connect it back to the broader dataset-level concept.

These examples show that the annotation and extraction pipelines often capture different taxonomic signals: coarse relevance labels, richness summaries, vernacular names, scientific names, and explicit taxon lists.

The working hypothesis tested here is that we should compare several parallel taxonomic views instead of relying on the raw `species` field alone:

- raw `species` strings with the existing enhanced matcher,
- derived `species_stripped_richness` to compare only the non-numeric delimited fragments that remain after dropping richness-bearing pieces entirely,
- derived `gbif_key_stripped_richness` to compare that same filtered view after resolving the leftover names to GBIF backbone keys,
- derived `taxon_richness_counts` to compare count-bearing GT strings against enumerated predictions,
- derived `taxon_richness_group_keys` to compare count + normalized group labels,
- derived `taxon_broad_group_labels` to compare broader group labels inferred from explicit mentions and GBIF hierarchy,
- derived `gbif_keys` to collapse vernacular vs scientific naming.


## Questions

1. Which mismatch patterns are evaluation artifacts rather than extraction failures?
2. Does a stripped-richness view isolate non-richness taxa when GT mixes count-bearing fragments with explicit taxa?
3. Does a count-focused derived field recover signal from cases like `73 weevil species` vs 73 enumerated taxa?
4. Does GBIF enrichment improve comparison for vernacular/scientific mismatches?
5. Which derived fields reflect dataset relevance better than the baseline `species` metric?


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from llm_metadata.schemas.data_paper import RunArtifact
from llm_metadata.schemas.fuster_features import DatasetFeaturesExtraction
from llm_metadata.taxonomy_eval import (
    DEFAULT_TAXONOMY_FIELDS,
    build_ground_truth_by_id,
    enrich_indexed_models,
    evaluate_taxonomy_fields,
)


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_GLOB = "*20260305_135009_dev_subset_pdf_file*.json"
GT_PATH = PROJECT_ROOT / "data/dataset_092624_validated.xlsx"

candidates = sorted((PROJECT_ROOT / "artifacts/runs").glob(RUN_GLOB))
candidates += sorted((PROJECT_ROOT / "data/prompt_eval_reports").glob(RUN_GLOB))
RUN_ARTIFACT_PATH = candidates[0] if candidates else PROJECT_ROOT / "data/prompt_eval_reports/20260305_135009_dev_subset_pdf_file.json"

if not RUN_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find run artifact. Update RUN_GLOB or RUN_ARTIFACT_PATH. Tried: {RUN_ARTIFACT_PATH}"
    )

try:
    artifact = RunArtifact.load_json(RUN_ARTIFACT_PATH)
    allowed_ids = [record.gt_record_id for record in artifact.records]
    pred_by_id = artifact.predictions_by_id(DatasetFeaturesExtraction)
except Exception:
    artifact = None
    raw_report = json.loads(RUN_ARTIFACT_PATH.read_text(encoding="utf-8"))
    allowed_ids = [int(record_id) for record_id in raw_report["records"].keys()]
    pred_rows = {}
    for row in raw_report["field_results"]:
        record_id = str(row["record_id"])
        pred_rows.setdefault(record_id, {})[row["field"]] = row["pred_value"]
    pred_by_id = {
        record_id: DatasetFeaturesExtraction.model_validate(values)
        for record_id, values in pred_rows.items()
    }

true_by_id = build_ground_truth_by_id(GT_PATH, allowed_ids=allowed_ids)

display(Markdown(
    f"**Run artifact:** `{RUN_ARTIFACT_PATH}`  \n**Records loaded:** GT={len(true_by_id)} | Pred={len(pred_by_id)}"
))


**Run artifact:** `C:\Users\beav3503\dev\llm_metadata\data\prompt_eval_reports\20260305_135009_dev_subset_pdf_file.json`  
**Records loaded:** GT=30 | Pred=30

## Proposed Solution Under Test

The proposal is not to replace the raw `species` field. It is to keep `DatasetFeaturesExtraction` as the prediction contract, then derive additional comparison views on `DatasetFeaturesEvaluation` objects:

- `parsed_species`: structured analysis view only,
- `taxon_richness_mentions`: structured count + group mentions,
- `species_stripped_richness`: enhanced-species comparison after splitting on common delimiters and dropping any fragment that contains numbers,
- `gbif_key_stripped_richness`: the same filtered view, but converted to exact-match GBIF keys when resolution succeeds,
- `taxon_richness_counts`: exact comparison on projected counts,
- `taxon_richness_group_keys`: exact comparison on projected `count|group` keys,
- `taxon_broad_group_labels`: exact comparison on broader relevance-oriented group labels,
- `gbif_keys`: exact comparison on resolved taxon identifiers.

This keeps the extraction contract stable while making evaluation more aligned with the annotation intent.


In [3]:
FIELDS = [
    "species",
    "species_stripped_richness",
    "gbif_key_stripped_richness",
    "taxon_richness_counts",
    "taxon_richness_group_keys",
    "taxon_broad_group_labels",
    "gbif_keys",
]

# Set include_gbif=False if you want to avoid live GBIF lookups or rely only on cached calls.
report = evaluate_taxonomy_fields(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=FIELDS,
    include_gbif=True,
    gbif_confidence_threshold=80,
)

metrics_df = report.metrics_to_pandas().sort_values(["f1", "precision", "recall"], ascending=False)
display(Markdown("### Strategy metrics\nThis dataframe is calculated from `report.metrics_to_pandas()` after running each field strategy over the GT/prediction pairs that are applicable for that strategy. Each row aggregates record-level `tp`, `fp`, `fn`, and `tn` counts for one comparison view, then derives precision, recall, F1, accuracy, and exact-match rate from those totals. For stripped-richness strategies, `n` is smaller than the full run when records are skipped as not applicable. `species_stripped_richness` is now GT-gated: it is only scored when the ground-truth side retains stripped residue. Sorting by F1 puts the strongest strategy for this run at the top."))
display(metrics_df)


### Strategy metrics
This dataframe is calculated from `report.metrics_to_pandas()` after running each field strategy over the GT/prediction pairs that are applicable for that strategy. Each row aggregates record-level `tp`, `fp`, `fn`, and `tn` counts for one comparison view, then derives precision, recall, F1, accuracy, and exact-match rate from those totals. For stripped-richness strategies, `n` is smaller than the full run when records are skipped as not applicable. `species_stripped_richness` is now GT-gated: it is only scored when the ground-truth side retains stripped residue. Sorting by F1 puts the strongest strategy for this run at the top.

,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
6,gbif_key_stripped_richness,19,4,2,0,13,0.826087,0.904762,0.863636,1.461538,0.769231
2,species_stripped_richness,34,47,5,0,23,0.419753,0.871795,0.566667,1.478261,0.565217
3,taxon_broad_group_labels,13,16,6,7,30,0.448276,0.684211,0.541667,0.666667,0.566667
4,taxon_richness_counts,16,14,16,0,30,0.533333,0.500000,0.516129,0.533333,0.533333
1,species,36,244,16,0,30,0.128571,0.692308,0.216867,1.200000,0.466667
0,gbif_keys,19,246,2,0,30,0.071698,0.904762,0.132867,0.633333,0.333333
5,taxon_richness_group_keys,0,3,12,19,30,0.000000,0.000000,NaN,0.633333,0.633333


In [4]:
true_enriched = enrich_indexed_models(true_by_id, include_gbif=True)
pred_enriched = enrich_indexed_models(pred_by_id, include_gbif=True)
detail_df = report.to_pandas()

species_mismatches = detail_df[(detail_df["field"] == "species") & (~detail_df["match"])]
stripped_species_mismatches = detail_df[(detail_df["field"] == "species_stripped_richness") & (~detail_df["match"])]
gbif_key_stripped_mismatches = detail_df[(detail_df["field"] == "gbif_key_stripped_richness") & (~detail_df["match"])]
richness_mismatches = detail_df[(detail_df["field"] == "taxon_richness_counts") & (~detail_df["match"])]
group_mismatches = detail_df[(detail_df["field"] == "taxon_richness_group_keys") & (~detail_df["match"])]
broad_group_mismatches = detail_df[(detail_df["field"] == "taxon_broad_group_labels") & (~detail_df["match"])]

display(Markdown("### Baseline species mismatches\nThis dataframe is calculated by filtering `detail_df = report.to_pandas()` to `field == 'species'` and `match == False`. It shows the records where the baseline enhanced-species matcher still disagrees after comparing the raw GT and predicted `species` lists, with row-level `tp`, `fp`, and `fn` counts showing how much overlap remained."))
display(species_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))

display(Markdown("### Stripped-richness species mismatches\nThis dataframe is calculated by filtering `detail_df` to `field == 'species_stripped_richness'` and `match == False`. Each raw species string is first split on common delimiters, then any fragment containing a number is dropped entirely. The remaining fragments are compared with the enhanced species matcher. This strategy is only applied when the ground-truth side retains non-richness residue, so records where GT strips to nothing do not appear here."))
display(stripped_species_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))

display(Markdown("### GBIF-key stripped-richness mismatches\nThis dataframe is calculated by filtering `detail_df` to `field == 'gbif_key_stripped_richness'` and `match == False`. It uses the same filtered non-numeric fragments as `species_stripped_richness`, then resolves those leftover names to GBIF backbone keys and compares the resulting key lists exactly. This strategy is only applied when both sides resolve to at least one key, so one-sided or unresolved residue is treated as not applicable rather than as an error."))
display(gbif_key_stripped_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))

display(Markdown("### Count-based mismatches\nThis dataframe is calculated by filtering the same detail table to `field == 'taxon_richness_counts'` and `match == False`. Before evaluation, both GT and predictions are projected into count-only richness values, so this table isolates cases where the count signal itself still does not line up even after ignoring taxon identity."))
display(richness_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))

display(Markdown("### Count+group mismatches\nThis dataframe is calculated by filtering `detail_df` to `field == 'taxon_richness_group_keys'` and `match == False`. For this strategy, each side is first normalized into `count|group` keys such as `73|weevil`, so the rows here show where an exact comparison on both richness and coarse group label still fails."))
display(group_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))

display(Markdown("### Broad-group mismatches\nThis dataframe is calculated by filtering `detail_df` to `field == 'taxon_broad_group_labels'` and `match == False`. The broad-group strategy maps explicit group mentions and GBIF hierarchy information into coarse labels like `bird`, `beetle`, or `mammal`, so these rows show where even that looser relevance-oriented projection still disagrees."))
display(broad_group_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(30))


### Baseline species mismatches
This dataframe is calculated by filtering `detail_df = report.to_pandas()` to `field == 'species'` and `match == False`. It shows the records where the baseline enhanced-species matcher still disagrees after comparing the raw GT and predicted `species` lists, with row-level `tp`, `fp`, and `fn` counts showing how much overlap remained.

,record_id,true_value,pred_value,tp,fp,fn
27,123,[Caribou],"[Rangifer tarandus, migratory tundra caribou, ...",1,1,0
33,175,"[sapin baumier, epinette blanche, bouleau jaun...","[Abies balsamea (L.) Mill., Betula alleghanien...",5,12,1
52,208,[c.132 species of benthic community],"[132 taxa (8 phyla) — dominance of arthropods,...",0,19,1
57,225,"[Benthic intertidal community (c.20 algae, c.1...","[Fucus distichus edentatus, Fucus vesiculosus,...",0,10,4
64,24,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), 953 in...",1,1,0
77,258,"[16 damselfly species, water mite]","[Amphiagrion saucium, Chromagrion conditum, Co...",1,16,1
83,27,"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",2,2,0
89,271,[11 mite species],"[Amblyseius fallacis (Garman), Typhlodromus ca...",0,13,1
95,29,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...",4,2,0
102,30,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,1,1,0


### Stripped-richness species mismatches
This dataframe is calculated by filtering `detail_df` to `field == 'species_stripped_richness'` and `match == False`. Each raw species string is first split on common delimiters, then any fragment containing a number is dropped entirely. The remaining fragments are compared with the enhanced species matcher. This strategy is only applied when the ground-truth side retains non-richness residue, so records where GT strips to nothing do not appear here.

,record_id,true_value,pred_value,tp,fp,fn
28,123,[Caribou],"[Rangifer tarandus, migratory tundra caribou, ...",1,1,0
34,175,"[sapin baumier, epinette blanche, bouleau jaun...","[Abies balsamea (L.) Mill, Betula alleghaniens...",5,12,1
40,19,[black-legged tick],None,0,0,1
58,225,[others)],"[Fucus distichus edentatus, Fucus vesiculosus,...",0,10,1
65,24,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), analysed]",1,1,0
78,258,[water mite],"[Amphiagrion saucium, Chromagrion conditum, Co...",0,17,1
84,27,"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",2,2,0
96,29,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...",4,3,0
103,30,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,1,1,0
131,37,"[Populus balsamifera, Populus deltoides, hybrids]","[Populus balsamifera, Populus deltoides, Popul...",2,0,1


### GBIF-key stripped-richness mismatches
This dataframe is calculated by filtering `detail_df` to `field == 'gbif_key_stripped_richness'` and `match == False`. It uses the same filtered non-numeric fragments as `species_stripped_richness`, then resolves those leftover names to GBIF backbone keys and compares the resulting key lists exactly. This strategy is only applied when both sides resolve to at least one key, so one-sided or unresolved residue is treated as not applicable rather than as an error.

...

,record_id,true_value,pred_value,tp,fp,fn
4,101,[1],[49],0,1,1
30,123,[1],[4],0,1,1
36,175,[6],[17],0,1,1
60,225,[15],[10],0,1,1
67,24,[1],[953],0,1,1
80,258,[16],[17],0,1,1
86,27,[2],[4],0,1,1
91,271,[11],[13],0,1,1
98,29,[4],[6],0,1,1
105,30,[1],[2],0,1,1


### Count+group mismatches
This dataframe is calculated by filtering `detail_df` to `field == 'taxon_richness_group_keys'` and `match == False`. For this strategy, each side is first normalized into `count|group` keys such as `73|weevil`, so the rows here show where an exact comparison on both richness and coarse group label still fails.

,record_id,true_value,pred_value,tp,fp,fn
5,101,None,[49|female caribou],0,1,0
55,208,[132|benthic community],"[132|— dominance of arthropods, mollusks, and ...",0,1,1
61,225,"[15|annelid, 15|mollusc]",None,0,0,2
68,24,None,[953|individuals sampled; 913 individuals geno...,0,1,0
81,258,[16|damselfly],None,0,0,1
92,271,[11|mite],None,0,0,1
117,315,[30|beetle],None,0,0,1
122,343,[30|bird],None,0,0,1
127,351,[73|weevil],None,0,0,1
139,38,"[12|mammal, 199|ground-dwelling beetle, 240|fl...",None,0,0,3


### Broad-group mismatches
This dataframe is calculated by filtering `detail_df` to `field == 'taxon_broad_group_labels'` and `match == False`. The broad-group strategy maps explicit group mentions and GBIF hierarchy information into coarse labels like `bird`, `beetle`, or `mammal`, so these rows show where even that looser relevance-oriented projection still disagrees.

,record_id,true_value,pred_value,tp,fp,fn
3,101,None,"[female caribou, mammal]",0,2,0
9,104,None,[mammal],0,1,0
29,123,None,[mammal],0,1,0
41,19,None,[tick],0,1,0
53,208,[benthic community],"[annelid, arthropod, mollusc, — dominance of a...",0,4,1
59,225,"[annelid, mollusc]","[arthropod, mollusc]",1,1,1
66,24,None,[individuals sampled; 913 individuals genotype...,0,1,0
79,258,[damselfly],"[arthropod, mite]",0,2,1
85,27,None,[mammal],0,1,0
125,351,[weevil],"[beetle, weevil]",1,1,0


In [5]:
records = []
for record_id in sorted(set(true_enriched) | set(pred_enriched), key=int):
    true_model = true_enriched.get(record_id)
    pred_model = pred_enriched.get(record_id)
    records.append(
        {
            "record_id": record_id,
            "true_species": getattr(true_model, "species", None),
            "pred_species": getattr(pred_model, "species", None),
            "true_richness_mentions": getattr(true_model, "taxon_richness_mentions", None),
            "pred_richness_mentions": getattr(pred_model, "taxon_richness_mentions", None),
            "true_richness_counts": getattr(true_model, "taxon_richness_counts", None),
            "pred_richness_counts": getattr(pred_model, "taxon_richness_counts", None),
            "true_group_keys": getattr(true_model, "taxon_richness_group_keys", None),
            "pred_group_keys": getattr(pred_model, "taxon_richness_group_keys", None),
            "true_broad_groups": getattr(true_model, "taxon_broad_group_labels", None),
            "pred_broad_groups": getattr(pred_model, "taxon_broad_group_labels", None),
            "true_species_stripped_richness": getattr(true_model, "species_stripped_richness", None),
            "pred_species_stripped_richness": getattr(pred_model, "species_stripped_richness", None),
            "true_gbif_key_stripped_richness": getattr(true_model, "gbif_key_stripped_richness", None),
            "pred_gbif_key_stripped_richness": getattr(pred_model, "gbif_key_stripped_richness", None),
        }
    )

taxonomy_view_df = pd.DataFrame.from_records(records)
display(Markdown("### Side-by-side taxonomic view table\nThis dataframe is calculated by enriching GT and prediction models first, then joining the raw `species` field with the derived taxonomic views for each `record_id`. It is an inspection table rather than a score table, and it lets you see how the same record looks under the raw, count-based, and broad-group representations on both sides."))
display(taxonomy_view_df.head(20))


### Side-by-side taxonomic view table
This dataframe is calculated by enriching GT and prediction models first, then joining the raw `species` field with the derived taxonomic views for each `record_id`. It is an inspection table rather than a score table, and it lets you see how the same record looks under the raw, count-based, and broad-group representations on both sides.

,record_id,true_species,pred_species,true_richness_mentions,pred_richness_mentions,true_richness_counts,pred_richness_counts,true_group_keys,pred_group_keys,true_broad_groups,pred_broad_groups,true_species_stripped_richness,pred_species_stripped_richness,true_gbif_key_stripped_richness,pred_gbif_key_stripped_richness
0,5,[Glyptemys insculpta],[Glyptemys insculpta],None,None,[1],[1],None,None,None,None,[Glyptemys insculpta],[Glyptemys insculpta],[2443115],[2443115]
1,9,"[raccoons, striped skunks]","[Procyon lotor (raccoon), Mephitis mephitis (s...",None,None,[2],[2],None,None,None,[mammal],"[raccoons, striped skunks]","[Procyon lotor (raccoon), Mephitis mephitis (s...",None,"[5218786, 5219380]"
2,11,[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],None,None,[1],[1],None,None,[mammal],[mammal],[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],[5220115],[5220115]
3,12,[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],None,None,[1],[1],None,None,[mammal],[mammal],[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],[5220115],[5220115]
4,19,[black-legged tick],[Ixodes scapularis Say 1821 (black-legged tick)],None,None,[1],[1],None,None,None,[tick],[black-legged tick],None,None,None
5,24,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), 953 in...",None,[original='953 individuals sampled; 913 indivi...,[1],[953],None,[953|individuals sampled; 913 individuals geno...,None,[individuals sampled; 913 individuals genotype...,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), analysed]",[3189859],[3189859]
6,27,"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",None,None,[2],[4],None,None,None,[mammal],"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",None,"[5219153, 5219200, 8441206]"
7,29,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...",None,None,[4],[6],None,None,None,None,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...","[2882813, 2882984, 2883007, 5333436]","[2882813, 2882868, 2882880, 2882984, 2883007, ..."
8,30,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,None,None,[1],[2],None,None,[mammal],[mammal],[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,[5220114],[5220115]
9,31,[Atlantic salmon],[Atlantic salmon (Salmo salar)],None,None,[1],[1],None,None,None,None,[Atlantic salmon],[Atlantic salmon (Salmo salar)],None,[7595433]


In [6]:
field_match_by_record = (
    detail_df.pivot_table(index="record_id", columns="field", values="match", aggfunc="max")
    .fillna(False)
    .astype(bool)
)
field_tp_by_record = (
    detail_df.pivot_table(index="record_id", columns="field", values="tp", aggfunc="sum")
    .fillna(0)
)

for field in FIELDS:
    if field not in field_match_by_record.columns:
        field_match_by_record[field] = False
    if field not in field_tp_by_record.columns:
        field_tp_by_record[field] = 0

species_mismatch_mask = ~field_match_by_record["species"]
stripped_species_match_mask = field_tp_by_record["species_stripped_richness"] > 0
gbif_key_stripped_match_mask = field_tp_by_record["gbif_key_stripped_richness"] > 0
group_key_match_mask = field_tp_by_record["taxon_richness_group_keys"] > 0
count_match_mask = field_tp_by_record["taxon_richness_counts"] > 0
broad_match_mask = field_tp_by_record["taxon_broad_group_labels"] > 0
gbif_match_mask = field_tp_by_record["gbif_keys"] > 0

analysis_bucket = pd.Series("No species mismatch", index=field_match_by_record.index)
analysis_bucket.loc[species_mismatch_mask] = "Unresolved after derived views"
analysis_bucket.loc[species_mismatch_mask & stripped_species_match_mask] = "Recovered by stripped richness"
analysis_bucket.loc[species_mismatch_mask & gbif_key_stripped_match_mask] = "Recovered by GBIF-key stripped richness"
analysis_bucket.loc[species_mismatch_mask & gbif_match_mask] = "Recovered by GBIF only"
analysis_bucket.loc[species_mismatch_mask & broad_match_mask] = "Recovered by broad groups"
analysis_bucket.loc[species_mismatch_mask & count_match_mask] = "Recovered by counts"
analysis_bucket.loc[species_mismatch_mask & count_match_mask & broad_match_mask] = "Recovered by counts + broad groups"
analysis_bucket.loc[species_mismatch_mask & group_key_match_mask] = "Recovered by count + group keys"

recovery_summary_df = (
    analysis_bucket[species_mismatch_mask]
    .rename("analysis_bucket")
    .value_counts()
    .rename_axis("analysis_bucket")
    .reset_index(name="records")
)
recovery_summary_df["share_of_species_mismatches"] = (
    recovery_summary_df["records"] / max(int(species_mismatch_mask.sum()), 1)
).round(3)

analysis_examples_df = field_match_by_record.reset_index().copy()
analysis_examples_df["analysis_bucket"] = analysis_bucket.values
analysis_examples_df["record_id"] = analysis_examples_df["record_id"].astype(str)
taxonomy_examples_df = taxonomy_view_df.copy()
taxonomy_examples_df["record_id"] = taxonomy_examples_df["record_id"].astype(str)

analysis_examples_df = analysis_examples_df.merge(taxonomy_examples_df, on="record_id", how="left")
analysis_examples_df = analysis_examples_df[
    analysis_examples_df["analysis_bucket"] != "No species mismatch"
].sort_values(["analysis_bucket", "record_id"])

display(Markdown("### Species-mismatch recovery summary\nThis dataframe is calculated by starting from records where the baseline `species` strategy failed, then assigning each record to the first derived-view bucket that shows positive overlap (`tp > 0`). Empty-field agreement does not count as recovery. The `share_of_species_mismatches` column is the fraction of all baseline species mismatches that fall into each bucket."))
display(recovery_summary_df)

display(Markdown("### Example records by recovery bucket\nThis dataframe is calculated by merging the recovery buckets back onto the side-by-side taxonomic view table. It shows representative records for each bucket so you can inspect which raw and derived signals caused a baseline `species` mismatch to remain unresolved or to be partially recovered by a derived strategy."))
display(
    analysis_examples_df[
        [
            "record_id",
            "analysis_bucket",
            "true_species",
            "pred_species",
            "true_species_stripped_richness",
            "pred_species_stripped_richness",
            "true_gbif_key_stripped_richness",
            "pred_gbif_key_stripped_richness",
            "true_richness_counts",
            "pred_richness_counts",
            "true_broad_groups",
            "pred_broad_groups",
        ]
    ].head(20)
)


C:\Users\beav3503\AppData\Local\Temp\ipykernel_27204\1743996926.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


### Species-mismatch recovery summary
This dataframe is calculated by starting from records where the baseline `species` strategy failed, then assigning each record to the first derived-view bucket that shows positive overlap (`tp > 0`). Empty-field agreement does not count as recovery. The `share_of_species_mismatches` column is the fraction of all baseline species mismatches that fall into each bucket.

,analysis_bucket,records,share_of_species_mismatches
0,Recovered by broad groups,6,0.375
1,Recovered by stripped richness,3,0.188
2,Recovered by counts,2,0.125
3,Recovered by GBIF only,2,0.125
4,Unresolved after derived views,2,0.125
5,Recovered by counts + broad groups,1,0.062


### Example records by recovery bucket
This dataframe is calculated by merging the recovery buckets back onto the side-by-side taxonomic view table. It shows representative records for each bucket so you can inspect which raw and derived signals caused a baseline `species` mismatch to remain unresolved or to be partially recovered by a derived strategy.

,record_id,analysis_bucket,true_species,pred_species,true_species_stripped_richness,pred_species_stripped_richness,true_gbif_key_stripped_richness,pred_gbif_key_stripped_richness,true_richness_counts,pred_richness_counts,true_broad_groups,pred_broad_groups
10,24,Recovered by GBIF only,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), 953 in...",[Acer saccharum],"[Acer saccharum Marshall (sugar maple), analysed]",[3189859],[3189859],[1],[953],None,[individuals sampled; 913 individuals genotype...
15,29,Recovered by GBIF only,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...","[2882813, 2882984, 2883007, 5333436]","[2882813, 2882868, 2882880, 2882984, 2883007, ...",[4],[6],None,None
9,225,Recovered by broad groups,"[Benthic intertidal community (c.20 algae, c.1...","[Fucus distichus edentatus, Fucus vesiculosus,...",[others)],"[Fucus distichus edentatus, Fucus vesiculosus,...",None,"[1, 229, 2285679, 2301175, 3197892, 3199693, 6...",[15],[10],"[annelid, mollusc]","[arthropod, mollusc]"
14,271,Recovered by broad groups,[11 mite species],"[Amblyseius fallacis (Garman), Typhlodromus ca...",None,"[Amblyseius fallacis (Garman), Typhlodromus ca...",None,"[2128625, 2131874, 2186588, 2186608, 2186891, ...",[11],[13],[mite],[mite]
16,30,Recovered by broad groups,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,[5220114],[5220115],[1],[2],[mammal],[mammal]
18,315,Recovered by broad groups,[c.30 beetle species],"[Histeridae, Hydrophilidae, Leiodidae, Ptilida...",None,"[Histeridae, Hydrophilidae, Leiodidae, Ptilida...",None,"[4733, 4738, 4762, 5839, 5840, 7830, 7854, 103...",[30],[29],[beetle],[beetle]
19,343,Recovered by broad groups,[c.30 bird species],"[Setophaga ruticilla, Setophaga castanea, Poec...",None,"[Setophaga ruticilla, Setophaga castanea, Poec...",None,"[2473492, 2473702, 2481710, 2481784, 2481798, ...",[30],[37],[bird],[bird]
22,38,Recovered by broad groups,"[12 mammal, 199 ground-dwelling beetles, 240 f...","[Tamiasciurus hudsonicus (red squirrel), Tamia...",None,"[Tamiasciurus hudsonicus (red squirrel), Tamia...",None,"[2435935, 2437282, 2437438, 2437967, 2438667, ...","[12, 199, 240]",[9],"[flying-beetle, ground-dwelling beetle, mammal]",[mammal]
8,208,Recovered by counts,[c.132 species of benthic community],"[132 taxa (8 phyla) — dominance of arthropods,...",None,"[mollusks, annelids, Micronephthys neotena, Eu...",None,"[1, 8002, 2215596, 2215896, 2215897, 2217213, ...",[132],[132],[benthic community],"[annelid, arthropod, mollusc, — dominance of a..."
21,37,Recovered by counts,"[Populus balsamifera, Populus deltoides, hybrids]","[Populus balsamifera, Populus deltoides, Popul...","[Populus balsamifera, Populus deltoides, hybrids]","[Populus balsamifera, Populus deltoides, Popul...","[3040184, 3040232]","[3040184, 3040232]",[3],[3],None,None


## Analysis

For this run, the derived taxonomic views are useful, but they answer different questions and should not be collapsed into a single replacement metric. The applicability masks now matter as much as the headline F1 values: some strategies are still scored on all 30 records, while others are scored only on the subset where that comparison is actually defined.

- `gbif_key_stripped_richness` now has the highest F1 in the notebook, but that score is on only 13 applicable records. It is answering a very specific question: after dropping every numeric fragment, do the leftover names resolve to the same GBIF backbone taxa on both sides? That makes it strong for measuring agreement on explicit non-richness residue, not for ranking overall taxonomic relevance by itself.
- `species_stripped_richness` is now much easier to interpret because it is GT-gated. Its F1 rises to `0.567` on `23` applicable records once count-only GT rows are removed from the denominator, which is the right behavior for a residue-only strategy. On this run it becomes the strongest non-key residue view, but it still inherits noise from leftover non-taxon fragments such as `others)` or `analysed`.
- `taxon_broad_group_labels` remains the strongest relevance-oriented view among strategies that still cover all 30 records. That matters: broad-group matching is looser than the stripped-residue strategies, but it continues to capture the screening question "is this dataset about the right kind of organism?" across the full subset rather than only on applicable residue-bearing cases. If one taxonomic score had to stand in for dataset-level relevance on this slice, this is the most defensible candidate.
- `taxon_richness_counts` recovers a different signal: the annotation sometimes encodes richness as a count-bearing group mention, while the model responds with an explicit list of taxa. The count projection helps with those cases, but it does not tell us whether the predicted taxa belong to the right group, so it should be read as a count-alignment check rather than a relevance score.
- `taxon_richness_group_keys` remains too strict for current model behavior. In most failures, the model enumerates species or gives partial counts, but does not emit a normalized count-plus-group phrase that lines up exactly with the ground truth representation.
- `gbif_keys` mostly helps when the disagreement is naming style rather than relevance, such as vernacular versus scientific names. It does not fix coarse-group GT strings, and its precision stays low when the model enumerates many taxa that are not represented in the annotation.
- The derived fields therefore work best as complementary diagnostic views. They show whether a baseline `species` miss is really a taxonomic relevance hit, a naming mismatch, a richness mismatch, or a genuine extraction error. The notebook should now be read as a coverage-aware comparison, not just a leaderboard of F1 values.

## Discussion

The current results support keeping raw `species` evaluation, but not treating it as the only taxonomic score. They also support applying strategy-specific applicability rules directly in evaluation rather than counting undefined comparisons as errors. Once that change is made, the stripped-richness strategies become interpretable instead of being dragged down by records they were never designed to score.

A practical reading of this run is: use `taxon_broad_group_labels` when the question is whole-dataset screening relevance, use `species` and `gbif_keys` when the question is literal taxon recovery or naming normalization, and use the stripped-richness strategies as narrower diagnostics for mixed richness-plus-name annotations. Those are not competing replacements for one another; they are instruments for different failure modes.

The main unresolved cases fall into two buckets:

- the model predicts many valid taxa, but the annotation is much coarser than any exact derived representation can capture;
- the model emits extra biological detail or non-taxon phrases, which broad-group projection can sometimes over-credit.

That implies three concrete next steps for prompt and evaluation work:

- keep reporting applicability alongside F1 for the stripped-richness strategies, because `0.864 on 13 applicable records` and `0.542 on 30 records` answer different questions and should not be read as a simple winner/loser comparison;
- clean the stripped-residue projection further so non-taxon leftovers such as `others)` and `analysed` do not inflate false positives in the residue-based strategies;
- if the adjudication supports it, either add a standard broad-group comparison view to `prompt_eval` reports or prompt the model to emit structured taxon objects directly so count and group identity can be evaluated without heavy post-processing.

Open questions to revisit after more runs:

- Which baseline `species` mismatches become reasonable matches under `species_stripped_richness` when GT mixes richness shorthand with one or two explicit taxa?
- Which baseline `species` mismatches become reasonable matches under `taxon_richness_counts` across abstract vs full-text modes?
- Should the metrics table surface an explicit `applicable_n` label or coverage percentage to reduce the chance of over-reading subset-only F1 improvements?
- Does `taxon_broad_group_labels` stay useful after auditing false positives from GBIF-to-group mapping?
- How often does `gbif_key_stripped_richness` add value beyond raw stripped richness once broad-group matching is available?


Personnal comments (Do not edit)

There many different signals captured in the groundtruth species values :
* Species richness : Number of organisme associated to a taxonomic group or habitat community. As single value or list.
* Species/taxonomic identity : Vernacular or scientific name of an organism, sometimes both. As single value or list.
* Group : Taxonomic group or habitat specific community. From species annotation, can be either isolated or joint to richness. As single value or list.

Right now : Both true and preds are mixed bags. Current investigation helped uncover eval strategy and relevant enrichment for each.

Road to success :

Taxonomic signal decoupled using distinct evaluated fields, either from llm extraction and/or species processing. Single evaluation for each.

* Contracts : New Extraction and Evaluation models including evaluated taxonomy fields + construction methods from Features models. Difference between extraction and evaluation is added gbif_keys from species.
* Extraction : Improved prompt with signal specific instructions and positive and negative examples from previous extractions. Wired prompt_eval. Improved species prompt to remove individual mentions and clearer instructions from signal observation. Strip non-taxons goop. Taxonomic signals should not be considered mutually exclusive.
* Evals : Per taxonomic signal eval srat.
    - Species field kept with enhanced_species strat.
    - 3 signals : Applicability is key – we don't want to include outcomes when a signal is unapplicable. Might need to rethink how to trigger applicability though with new extracted features. Stripped_richness was an experiment, still unsure on how I feel about it. Solution might be on how I derive values from 
    - gbif_keys : I think it's useful. Again, applicability is key.